# Validação corrigida dos modelos

Experimento congelado; este notebook somente lê resultados. Nenhum treino, download ou compilação é executado.

Leia [o relatório](reports/validacao_corrigida_2026_10_01.md). Estado: experimental, generalização insuficiente.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'results/corrected_2026_10_01').exists():
    ROOT = ROOT.parent
O = ROOT / 'results/corrected_2026_10_01'
freeze = json.loads((O / 'freeze.json').read_text())
freeze

{'frozen_utc': '2026-10-01T19:22:43.947457+00:00',
 'models_sha256': {'results/corrected_2026_10_01/models_trees/lightgbm.joblib': '3c4710d71b011e2c33ffb14b08e66cded5bf4d0147f104782138bf2a811bcee7',
  'results/corrected_2026_10_01/models_trees/hist_gradient_boosting.joblib': '6dcfa3fac4e7c2a8b5d29f93340022a6c7a75bff18824a8bad517b68a26aea8e',
  'results/corrected_2026_10_01/models_trees/extra_trees.joblib': '1aca2d0f67475e7f54671dbb054eccfb134ff15512eb66bec6c49cd30f8dfa10',
  'results/corrected_2026_10_01/models_trees/random_forest.joblib': '3ad15ffe6b7348a3a365620089d63d26743a8ea4bdeca2fb9fa62a87ef508420',
  'results/corrected_2026_10_01/models_committee/committees_bundle.joblib': '42a5d9dffe2e4809d1165fd42e861f98e97897c6603a537249cf20361a121b2f',
  'results/corrected_2026_10_01/models_committee/corrected_bundle.joblib': 'b313cb36a025ceb69824f86767d7683f278a5cecefd60c51ae8b9d77f22c3f8c',
  'results/models_deep/deep_hybrid_net.pt': '376934c58916177fc741e036024d377eb6ac0c0eea5f296165c8e4

## Separação por gravação e estabilidade por lote

In [2]:
display(pd.read_json(O / 'models_trees/fold_audit.json'))
display(json.loads((O / 'batch_invariance_audit.json').read_text()))

,fold,train_windows,validation_windows,overlapping_groups,validation_groups
0,0,12688,4703,0,68
1,1,14220,3171,0,70
2,2,14221,3170,0,72
3,3,14218,3173,0,72
4,4,14217,3174,0,73


{'rows': 1575,
 'max_difference_single': 1.1102230246251565e-16,
 'max_difference_by_file': 0.0}

## Comparações históricas: sensibilidade, falsos positivos e rejeições

In [3]:
historical = pd.read_csv(O / 'historical_metrics.csv')
historical[['model','domain','windows','balanced_accuracy','emitted_aedes_recall','emitted_false_positive_rate','known_coverage']]

,model,domain,windows,balanced_accuracy,emitted_aedes_recall,emitted_false_positive_rate,known_coverage
0,tree_lightgbm,dryad,7901,0.910441,0.802778,0.037926,0.955196
1,tree_lightgbm,humbug,1510,0.921266,0.961039,0.074301,0.907550
2,tree_lightgbm,mosquitosong,352,NaN,0.571023,NaN,0.846591
3,tree_lightgbm,ufrgs,417,0.696937,0.372694,0.102740,0.717026
4,tree_hist_gradient_boosting,dryad,7901,0.907007,0.825000,0.037263,0.970383
5,tree_hist_gradient_boosting,humbug,1510,0.937812,0.961039,0.063811,0.940678
6,tree_hist_gradient_boosting,mosquitosong,352,NaN,0.585227,NaN,0.889205
7,tree_hist_gradient_boosting,ufrgs,417,0.704582,0.416974,0.102740,0.764988
8,tree_extra_trees,dryad,7901,0.906581,0.719444,0.036865,0.937350
9,tree_extra_trees,humbug,1510,0.937188,0.909091,0.033217,0.911402


## Misturas sintéticas: resultado por arquivo, sem garantia de origem independente

In [4]:
pd.read_csv(O / 'synthetic_file_metrics_exploratory.csv')

,model,windows,positive_windows,negative_windows,unknown_windows,invalid,auc,forced_threshold,balanced_accuracy,forced_accuracy,aedes_emissions,true_positive_emissions,false_positive_emissions,emitted_aedes_recall,emitted_false_positive_rate,emitted_precision,false_negative_emissions,uncertain,known_coverage,unresolved_aedes_emissions
0,tree_lightgbm,315,100,215,0,0,0.559674,0.500000,0.546512,0.526984,50,18,32,0.18,0.148837,0.360000,19,195,0.380952,0
1,tree_hist_gradient_boosting,315,100,215,0,0,0.546837,0.500000,0.526163,0.517460,54,21,33,0.21,0.153488,0.388889,26,171,0.457143,0
2,tree_extra_trees,315,100,215,0,0,0.523209,0.500000,0.511628,0.479365,16,3,13,0.03,0.060465,0.187500,22,216,0.314286,0
3,tree_random_forest,315,100,215,0,0,0.498279,0.500000,0.508721,0.457143,9,1,8,0.01,0.037209,0.111111,11,273,0.133333,0
4,baseline_svm_rbf,315,100,215,0,0,0.651349,0.500000,0.632326,0.622222,11,5,6,0.05,0.027907,0.454545,1,297,0.057143,0
5,baseline_extra_trees,315,100,215,0,0,0.556791,0.500000,0.540349,0.485714,29,12,17,0.12,0.079070,0.413793,13,227,0.279365,0
6,baseline_gradient_boosting,315,100,215,0,0,0.572605,0.500000,0.536860,0.444444,88,33,55,0.33,0.255814,0.375000,3,217,0.311111,0
7,committee_soft_voting,315,100,215,0,0,0.538186,0.500000,0.516279,0.485714,39,14,25,0.14,0.116279,0.358974,27,177,0.438095,0
8,committee_stacking,315,100,215,0,0,0.481535,0.500000,0.494070,0.663492,4,0,4,0.00,0.018605,0.000000,0,310,0.015873,0
9,deep_hybrid_net,315,100,215,0,0,0.668465,0.500000,0.559767,0.698413,31,18,13,0.18,0.060465,0.580645,4,246,0.219048,0


## D5 óptico: limiar da validação separado do teste

In [5]:
d5 = pd.read_csv(O / 'd5_test_metrics.csv')
d5[d5.calibration.eq('separate_D5_validation')][['model','windows','auc','balanced_accuracy','emitted_aedes_recall','emitted_false_positive_rate']]

,model,windows,auc,balanced_accuracy,emitted_aedes_recall,emitted_false_positive_rate
1,tree_lightgbm,400,0.593900,0.5750,0.810,0.660
3,tree_hist_gradient_boosting,400,0.595550,0.5775,0.795,0.640
5,tree_extra_trees,400,0.616425,0.5950,0.830,0.640
7,tree_random_forest,400,0.608375,0.6125,0.685,0.460
9,baseline_svm_rbf,400,0.548075,0.5800,0.830,0.670
11,baseline_extra_trees,400,0.625000,0.6125,0.715,0.490
13,baseline_gradient_boosting,400,0.607500,0.5975,0.645,0.450
15,committee_soft_voting,400,0.609950,0.5875,0.800,0.625
17,committee_stacking,400,0.598525,0.5875,0.675,0.500
19,deep_hybrid_net,400,0.542850,0.5450,0.750,0.660


## Áudios acústicos inéditos após congelamento

Somente negativos: não permitem estimar sensibilidade em Aedes inédito. Há três gravações novas de Anopheles, com seis janelas, e 12 gravações novas de ruído, com 161 janelas.

In [6]:
pd.read_csv(O / 'fresh_metrics.csv')[['model','source','windows','aedes_emissions','emitted_false_positive_rate','uncertain']]

,model,source,windows,aedes_emissions,emitted_false_positive_rate,uncertain
0,baseline_extra_trees,fresh_noise,161,5,0.031056,34
1,baseline_extra_trees,fresh_other_mosquito,6,5,0.833333,0
2,baseline_gradient_boosting,fresh_noise,161,30,0.186335,81
3,baseline_gradient_boosting,fresh_other_mosquito,6,2,0.333333,3
4,baseline_svm_rbf,fresh_noise,161,14,0.086957,33
5,baseline_svm_rbf,fresh_other_mosquito,6,0,0.000000,4
6,committee_soft_voting,fresh_noise,161,6,0.037267,18
7,committee_soft_voting,fresh_other_mosquito,6,3,0.500000,2
8,committee_stacking,fresh_noise,161,2,0.012422,104
9,committee_stacking,fresh_other_mosquito,6,2,0.333333,4


## Áudio privado

Referência do usuário: Aedes em todo o áudio. Métrica de sensibilidade em uma gravação conhecida; não acurácia geral.

In [7]:
pd.read_csv(O / 'private_audio_aggregate.csv')[['model','windows','aedes_emissions','emitted_aedes_recall','false_negative_emissions','uncertain','invalid']]

,model,windows,aedes_emissions,emitted_aedes_recall,false_negative_emissions,uncertain,invalid
0,tree_lightgbm,107,38,0.355140,35,32,2
1,tree_hist_gradient_boosting,107,36,0.336449,45,24,2
2,tree_extra_trees,107,48,0.448598,30,27,2
3,tree_random_forest,107,32,0.299065,18,55,2
4,baseline_svm_rbf,107,25,0.233645,41,39,2
5,baseline_extra_trees,107,66,0.616822,26,13,2
6,baseline_gradient_boosting,107,55,0.514019,23,27,2
7,committee_soft_voting,107,42,0.392523,33,30,2
8,committee_stacking,107,18,0.168224,5,82,2
9,deep_hybrid_net,107,59,0.551402,32,14,2


## Incerteza por gravação e decisão de confiabilidade

In [8]:
display(pd.read_csv(O / 'recording_cluster_intervals.csv'))
display(json.loads((O / 'reliability_decision.json').read_text()))

,model,domain,metric,source_groups,windows,low95,high95,note
0,tree_lightgbm,dryad,recall,6,360,0.738889,0.883333,"Recording-cluster bootstrap; exploratory, sour..."
1,tree_lightgbm,dryad,fpr,198,7541,0.021364,0.058317,"Recording-cluster bootstrap; exploratory, sour..."
2,tree_hist_gradient_boosting,dryad,recall,6,360,0.761111,0.902778,"Recording-cluster bootstrap; exploratory, sour..."
3,tree_hist_gradient_boosting,dryad,fpr,198,7541,0.021273,0.057205,"Recording-cluster bootstrap; exploratory, sour..."
4,tree_extra_trees,dryad,recall,6,360,0.563889,0.875000,"Recording-cluster bootstrap; exploratory, sour..."
...,...,...,...,...,...,...,...,...
91,baseline_svm_rbf,ufrgs,fpr,1,146,NaN,NaN,Insufficient independent source groups; no int...
92,baseline_extra_trees,ufrgs,recall,2,271,0.717557,0.900000,"Recording-cluster bootstrap; exploratory, sour..."
93,baseline_extra_trees,ufrgs,fpr,1,146,NaN,NaN,Insufficient independent source groups; no int...
94,baseline_gradient_boosting,ufrgs,recall,2,271,0.541985,0.964286,"Recording-cluster bootstrap; exploratory, sour..."


{'status': 'EXPERIMENTAL_CROSS_SOURCE_VALIDATION_FAILURE',
 'models_released': False,
 'original_weights_preserved': True,
 'arduino_compiled': False,
 'reasons': ['Swarm ensemble remains near chance on Abuzz-derived historical mixtures.',
  'New optical test shows 54.5–61.25% balanced accuracy after separate calibration.',
  'Fresh non-Aedes recordings still produce false Aedes emissions.',
  'No unused positive HumBug groups remained; fresh acoustic sensitivity is unmeasured.']}